# Reinforcement Learning: Exercises

10 graded exercises covering MDP fundamentals, value-based methods, policy gradients, and modern alignment techniques. Each exercise has a problem statement, starter code, and a complete solution.

In [ ]:
import numpy as np
from typing import Tuple, List, Dict
np.random.seed(42)

## Exercise 1: Bellman Equation as a Linear System

Given a 4-state chain MDP with transition probabilities and rewards, solve the Bellman expectation equation for a uniform random policy.

**Tasks:**
1. Write the Bellman equation as $\mathbf{v} = \mathbf{r}^\pi + \gamma P^\pi \mathbf{v}$
2. Solve by matrix inversion: $\mathbf{v} = (I - \gamma P^\pi)^{-1} \mathbf{r}^\pi$
3. Verify by iterative policy evaluation

In [ ]:
# MDP: 4 states (0,1,2,3), state 3 is terminal
# Actions: left (0), right (1)
# Transitions: deterministic
# Rewards: -1 per step, +10 at terminal

n_states = 4
gamma = 0.99

# Transition matrix under uniform random policy
# P_pi[s, s'] = sum_a pi(a|s) * P(s'|s,a)
# With uniform policy: pi(left) = pi(right) = 0.5

# YOUR CODE HERE: construct P_pi and r_pi, then solve
P_pi = None  # shape (4, 4)
r_pi = None  # shape (4,)
V = None     # solve (I - gamma * P_pi) V = r_pi

# Also implement iterative policy evaluation
def policy_evaluation(P_pi, r_pi, gamma, theta=1e-10):
    V = np.zeros(n_states)
    # YOUR CODE HERE
    return V

### Solution 1

In [ ]:
n_states = 4
gamma = 0.99

# State 0: left->0, right->1
# State 1: left->0, right->2
# State 2: left->1, right->3
# State 3: terminal (absorbing)
# Uniform policy: 0.5 left + 0.5 right

P_pi = np.array([
    [0.5, 0.5, 0.0, 0.0],  # s0: 0.5*stay + 0.5*go_right
    [0.5, 0.0, 0.5, 0.0],  # s1: 0.5*go_left + 0.5*go_right
    [0.0, 0.5, 0.0, 0.5],  # s2: 0.5*go_left + 0.5*go_right
    [0.0, 0.0, 0.0, 1.0],  # s3: terminal (stay)
])

# Expected reward under uniform policy
# r_pi[s] = sum_a pi(a|s) sum_s' P(s'|s,a) R(s,a,s')
r_pi = np.array([-1.0, -1.0, 0.5*(-1.0) + 0.5*(10.0), 0.0])

# Direct solve
I = np.eye(n_states)
V_direct = np.linalg.solve(I - gamma * P_pi, r_pi)

# Iterative policy evaluation
def policy_evaluation(P_pi, r_pi, gamma, theta=1e-10):
    V = np.zeros(n_states)
    for _ in range(10000):
        V_new = r_pi + gamma * P_pi @ V
        if np.max(np.abs(V_new - V)) < theta:
            break
        V = V_new
    return V

V_iter = policy_evaluation(P_pi, r_pi, gamma)

print("Direct solve (matrix inversion):", V_direct.round(4))
print("Iterative policy evaluation:    ", V_iter.round(4))
print("Max difference:", np.max(np.abs(V_direct - V_iter)))

## Exercise 2: TD(0) vs Monte Carlo on Random Walk

Implement TD(0) and first-visit MC on a 5-state random walk.

**Tasks:**
1. Implement both prediction methods
2. Run 100 episodes, track RMSE vs true values
3. Compare convergence speed

In [ ]:
class RandomWalk:
    """States 1-5, terminal at 0 and 6. Start at 3."""
    def __init__(self):
        self.state = 3

    def reset(self):
        self.state = 3
        return self.state

    def step(self):
        self.state += 1 if np.random.random() > 0.5 else -1
        done = self.state == 0 or self.state == 6
        reward = 1.0 if self.state == 6 else 0.0
        return self.state, reward, done

true_V = np.array([0, 1/6, 2/6, 3/6, 4/6, 5/6, 0])

def td0(env, alpha=0.1, n_episodes=100):
    V = np.full(7, 0.5); V[0] = V[6] = 0
    rmse_history = []
    # YOUR CODE HERE
    return rmse_history

def mc_prediction(env, alpha=0.01, n_episodes=100):
    V = np.full(7, 0.5); V[0] = V[6] = 0
    rmse_history = []
    # YOUR CODE HERE
    return rmse_history

### Solution 2

In [ ]:
class RandomWalk:
    def __init__(self):
        self.state = 3
    def reset(self):
        self.state = 3
        return self.state
    def step(self):
        self.state += 1 if np.random.random() > 0.5 else -1
        done = self.state == 0 or self.state == 6
        reward = 1.0 if self.state == 6 else 0.0
        return self.state, reward, done

true_V = np.array([0, 1/6, 2/6, 3/6, 4/6, 5/6, 0])

def td0(env, alpha=0.1, n_episodes=100):
    V = np.full(7, 0.5); V[0] = V[6] = 0
    rmse_history = []
    for _ in range(n_episodes):
        s = env.reset()
        while True:
            ns, r, done = env.step()
            target = r if done else r + V[ns]
            V[s] += alpha * (target - V[s])
            s = ns
            if done: break
        rmse_history.append(np.sqrt(np.mean((V[1:6] - true_V[1:6])**2)))
    return rmse_history

def mc_prediction(env, alpha=0.01, n_episodes=100):
    V = np.full(7, 0.5); V[0] = V[6] = 0
    rmse_history = []
    for _ in range(n_episodes):
        s = env.reset()
        trajectory = [s]
        rewards = []
        while True:
            ns, r, done = env.step()
            rewards.append(r)
            trajectory.append(ns)
            if done: break
        G = 0
        for t in reversed(range(len(rewards))):
            G += rewards[t]
            V[trajectory[t]] += alpha * (G - V[trajectory[t]])
        rmse_history.append(np.sqrt(np.mean((V[1:6] - true_V[1:6])**2)))
    return rmse_history

# Average over multiple runs
n_runs = 100
td_rmses = np.mean([td0(RandomWalk()) for _ in range(n_runs)], axis=0)
mc_rmses = np.mean([mc_prediction(RandomWalk()) for _ in range(n_runs)], axis=0)

print(f"Final RMSE - TD(0): {td_rmses[-1]:.4f}")
print(f"Final RMSE - MC:    {mc_rmses[-1]:.4f}")
print(f"TD(0) converges {'faster' if td_rmses[-1] < mc_rmses[-1] else 'slower'} on this task")

## Exercise 3: Q-Learning on Cliff Walking

Implement Q-learning on the cliff walking environment.

**Tasks:**
1. Implement epsilon-greedy Q-learning
2. Track reward per episode
3. Show the learned optimal path

In [ ]:
class CliffWalking:
    """4x12 grid. Start: (3,0), Goal: (3,11). Cliff: bottom row middle."""
    def __init__(self):
        self.h, self.w = 4, 12
        self.start, self.goal = (3, 0), (3, 11)
        self.cliff = [(3, c) for c in range(1, 11)]
        self.state = self.start

    def reset(self):
        self.state = self.start
        return self.state

    def step(self, action):
        dy, dx = [(-1,0),(1,0),(0,-1),(0,1)][action]
        ny = max(0, min(self.h-1, self.state[0]+dy))
        nx = max(0, min(self.w-1, self.state[1]+dx))
        self.state = (ny, nx)
        if self.state in self.cliff:
            self.state = self.start
            return self.state, -100.0, False
        if self.state == self.goal:
            return self.state, 0.0, True
        return self.state, -1.0, False

# YOUR CODE HERE: implement Q-learning
def q_learning_cliff(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((env.h, env.w, 4))
    rewards = []
    # YOUR CODE HERE
    return Q, rewards

### Solution 3

In [ ]:
class CliffWalking:
    def __init__(self):
        self.h, self.w = 4, 12
        self.start, self.goal = (3, 0), (3, 11)
        self.cliff = [(3, c) for c in range(1, 11)]
        self.state = self.start
    def reset(self):
        self.state = self.start
        return self.state
    def step(self, action):
        dy, dx = [(-1,0),(1,0),(0,-1),(0,1)][action]
        ny = max(0, min(self.h-1, self.state[0]+dy))
        nx = max(0, min(self.w-1, self.state[1]+dx))
        self.state = (ny, nx)
        if self.state in self.cliff:
            self.state = self.start
            return self.state, -100.0, False
        if self.state == self.goal:
            return self.state, 0.0, True
        return self.state, -1.0, False

def q_learning_cliff(env, n_episodes=500, alpha=0.5, gamma=1.0, epsilon=0.1):
    Q = np.zeros((env.h, env.w, 4))
    rewards = []
    for ep in range(n_episodes):
        s = env.reset()
        total_r = 0
        for _ in range(1000):
            if np.random.random() < epsilon:
                a = np.random.randint(4)
            else:
                a = np.argmax(Q[s[0], s[1]])
            ns, r, done = env.step(a)
            best_next = np.max(Q[ns[0], ns[1]])
            Q[s[0], s[1], a] += alpha * (r + gamma * best_next * (1-done) - Q[s[0], s[1], a])
            total_r += r
            s = ns
            if done: break
        rewards.append(total_r)
    return Q, rewards

env = CliffWalking()
Q, rewards = q_learning_cliff(env)

print(f"Avg reward (last 50): {np.mean(rewards[-50:]):.1f}")

# Show optimal path
arrows = ['U', 'D', 'L', 'R']
print("\nLearned policy (row 2, above cliff):")
for x in range(12):
    a = np.argmax(Q[2, x])
    print(f"  ({2},{x}): {arrows[a]}", end="")
print()

## Exercise 4: Policy Gradient Theorem Derivation (Computational)

Verify the policy gradient theorem numerically.

**Tasks:**
1. Compute the policy gradient using finite differences
2. Compute it using the REINFORCE estimator
3. Verify they match (up to variance)

In [ ]:
# Simple 2-state, 2-action MDP
# Policy: softmax with single parameter theta
# pi(right | s=0) = sigmoid(theta)

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class TwoStateMDP:
    def __init__(self):
        self.state = 0
    def reset(self):
        self.state = 0
        return 0
    def step(self, action):
        if action == 1:  # right
            return 1, 1.0, True  # goal, reward 1
        else:
            return 0, -0.1, False  # stay

def policy_return(theta, n_samples=1000):
    """Estimate J(theta) via sampling."""
    total = 0
    for _ in range(n_samples):
        env = TwoStateMDP()
        s = env.reset()
        ret = 0
        for t in range(20):
            p_right = sigmoid(theta)
            a = 1 if np.random.random() < p_right else 0
            ns, r, done = env.step(a)
            ret += r
            s = ns
            if done: break
        total += ret
    return total / n_samples

# YOUR CODE HERE: compute gradient two ways
# 1. Finite differences: (J(theta+h) - J(theta-h)) / (2h)
# 2. REINFORCE: E[G * d/dtheta log pi]

### Solution 4

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

class TwoStateMDP:
    def __init__(self):
        self.state = 0
    def reset(self):
        self.state = 0
        return 0
    def step(self, action):
        if action == 1:
            return 1, 1.0, True
        else:
            return 0, -0.1, False

def policy_return(theta, n_samples=5000):
    total = 0
    for _ in range(n_samples):
        env = TwoStateMDP()
        s = env.reset()
        ret = 0
        for t in range(20):
            p_right = sigmoid(theta)
            a = 1 if np.random.random() < p_right else 0
            ns, r, done = env.step(a)
            ret += r
            s = ns
            if done: break
        total += ret
    return total / n_samples

# Method 1: Finite differences
theta = 0.0
h = 0.01
grad_fd = (policy_return(theta + h) - policy_return(theta - h)) / (2 * h)

# Method 2: REINFORCE
n_samples = 10000
grad_rf = 0
for _ in range(n_samples):
    env = TwoStateMDP()
    s = env.reset()
    log_probs = []
    rewards = []
    for t in range(20):
        p = sigmoid(theta)
        a = 1 if np.random.random() < p else 0
        # d/dtheta log pi(a|s) for sigmoid policy
        if a == 1:
            log_grad = 1 - p  # d/dtheta log(sigmoid(theta))
        else:
            log_grad = -p     # d/dtheta log(1 - sigmoid(theta))
        log_probs.append(log_grad)
        ns, r, done = env.step(a)
        rewards.append(r)
        s = ns
        if done: break
    G = sum(rewards)
    grad_rf += sum(lg * G for lg in log_probs)
grad_rf /= n_samples

print(f"Finite differences gradient: {grad_fd:.4f}")
print(f"REINFORCE gradient:          {grad_rf:.4f}")
print(f"Difference: {abs(grad_fd - grad_rf):.4f}")
print("\nBoth estimate the same quantity (policy gradient theorem verified)")

## Exercise 5: PPO Clipping Analysis

Implement the PPO clipped objective and analyse its behaviour.

**Tasks:**
1. Implement $L^{CLIP}$ for given importance ratios and advantages
2. Compute the gradient (which regions are active vs clipped)
3. Show that clipping prevents large updates

In [ ]:
def ppo_clip_objective(ratios, advantages, epsilon=0.2):
    """Compute PPO clipped objective."""
    # YOUR CODE HERE
    pass

# Test cases
ratios = np.array([0.5, 0.8, 1.0, 1.2, 1.5, 2.0])
advantages = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.0])  # good actions
# What should happen: ratios > 1.2 should be clipped

### Solution 5

In [ ]:
def ppo_clip_objective(ratios, advantages, epsilon=0.2):
    """Compute PPO clipped objective element-wise."""
    unclipped = ratios * advantages
    clipped = np.clip(ratios, 1 - epsilon, 1 + epsilon) * advantages
    return np.minimum(unclipped, clipped)

# Test: A > 0 (good action, want to increase probability)
ratios = np.array([0.5, 0.8, 1.0, 1.2, 1.5, 2.0])
adv_pos = np.ones(6)  # A > 0

obj = ppo_clip_objective(ratios, adv_pos, epsilon=0.2)
print("A > 0 (good action):")
print(f"  Ratios:    {ratios}")
print(f"  Objective: {obj}")
print(f"  Note: r=1.5 and r=2.0 are clipped to 1.2")

# Test: A < 0 (bad action, want to decrease probability)
adv_neg = -np.ones(6)
obj_neg = ppo_clip_objective(ratios, adv_neg, epsilon=0.2)
print(f"\nA < 0 (bad action):")
print(f"  Ratios:    {ratios}")
print(f"  Objective: {obj_neg}")
print(f"  Note: r=0.5 is clipped to -0.8 (prevents over-suppression)")

# Gradient analysis
print("\nGradient is zero (clipped) when:")
print("  A > 0 and r > 1+eps: gradient stops (can't make action even more likely)")
print("  A < 0 and r < 1-eps: gradient stops (can't suppress action further)")

## Exercise 6: Double Q-Learning on Maximisation Bias MDP

Implement Double Q-learning and compare with standard Q-learning on the maximisation bias example.

**Tasks:**
1. Implement both algorithms
2. Track percentage of optimal action choices
3. Show that Double Q-learning reduces overestimation

In [ ]:
# The maximisation bias MDP:
# State A: left -> terminal (reward 0), right -> state B
# State B: 10 actions, each gives N(-0.1, 1) reward -> terminal
# Optimal: go left from A (expected return 0 > -0.1)

# YOUR CODE HERE: implement and compare
def standard_q(n_episodes=300):
    pass  # return list of (1 if chose left, 0 if right) per episode

def double_q(n_episodes=300):
    pass  # return same format

### Solution 6

In [ ]:
def standard_q(n_episodes=300, epsilon=0.1, alpha=0.1):
    Q_A = np.zeros(2)  # left, right
    Q_B = np.zeros(10)
    choices = []
    for ep in range(n_episodes):
        # State A
        a = np.random.randint(2) if np.random.random() < epsilon else np.argmax(Q_A)
        if a == 0:  # left -> terminal
            Q_A[0] += alpha * (0 - Q_A[0])
        else:       # right -> state B
            # State B
            b = np.random.randint(10) if np.random.random() < epsilon else np.argmax(Q_B)
            r = np.random.normal(-0.1, 1.0)
            Q_B[b] += alpha * (r - Q_B[b])
            Q_A[1] += alpha * (np.max(Q_B) - Q_A[1])  # overestimation here!
        choices.append(1 if np.argmax(Q_A) == 0 else 0)
    return choices

def double_q(n_episodes=300, epsilon=0.1, alpha=0.1):
    Q1_A, Q2_A = np.zeros(2), np.zeros(2)
    Q1_B, Q2_B = np.zeros(10), np.zeros(10)
    choices = []
    for ep in range(n_episodes):
        Q_sum = Q1_A + Q2_A
        a = np.random.randint(2) if np.random.random() < epsilon else np.argmax(Q_sum)
        if a == 0:
            if np.random.random() < 0.5:
                Q1_A[0] += alpha * (0 - Q1_A[0])
            else:
                Q2_A[0] += alpha * (0 - Q2_A[0])
        else:
            b_sum = Q1_B + Q2_B
            b = np.random.randint(10) if np.random.random() < epsilon else np.argmax(b_sum)
            r = np.random.normal(-0.1, 1.0)
            if np.random.random() < 0.5:
                Q1_B[b] += alpha * (r - Q1_B[b])
                Q1_A[1] += alpha * (Q2_B[np.argmax(Q1_B)] - Q1_A[1])
            else:
                Q2_B[b] += alpha * (r - Q2_B[b])
                Q2_A[1] += alpha * (Q1_B[np.argmax(Q2_B)] - Q2_A[1])
        choices.append(1 if np.argmax(Q1_A + Q2_A) == 0 else 0)
    return choices

# Run experiment
n_runs = 200
sq_results = np.mean([standard_q() for _ in range(n_runs)], axis=0)
dq_results = np.mean([double_q() for _ in range(n_runs)], axis=0)

print(f"% choosing optimal (left) in last 100 episodes:")
print(f"  Standard Q-learning: {100*np.mean(sq_results[-100:]):.1f}%")
print(f"  Double Q-learning:   {100*np.mean(dq_results[-100:]):.1f}%")

## Exercise 7: DPO Loss Computation

Implement the DPO loss given log-probabilities from policy and reference models.

$$\mathcal{L}_{DPO} = -\log \sigma\left(\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\right)$$

In [ ]:
def dpo_loss(log_pi_w, log_ref_w, log_pi_l, log_ref_l, beta=0.1):
    """
    Compute DPO loss.
    Args:
        log_pi_w: log pi_theta(y_w | x)  (winner)
        log_ref_w: log pi_ref(y_w | x)   (winner, reference)
        log_pi_l: log pi_theta(y_l | x)  (loser)
        log_ref_l: log pi_ref(y_l | x)   (loser, reference)
    """
    # YOUR CODE HERE
    pass

### Solution 7

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def dpo_loss(log_pi_w, log_ref_w, log_pi_l, log_ref_l, beta=0.1):
    log_ratio_w = log_pi_w - log_ref_w
    log_ratio_l = log_pi_l - log_ref_l
    logit = beta * (log_ratio_w - log_ratio_l)
    loss = -np.log(sigmoid(logit) + 1e-10)
    return loss

# Test cases
# Case 1: Policy correctly assigns higher prob to winner
loss1 = dpo_loss(
    log_pi_w=-1.0,   # pi gives winner high prob
    log_ref_w=-2.0,  # ref gives winner lower prob
    log_pi_l=-3.0,   # pi gives loser low prob
    log_ref_l=-2.0,  # ref gives loser moderate prob
    beta=1.0
)

# Case 2: Policy incorrectly prefers loser
loss2 = dpo_loss(
    log_pi_w=-3.0,   # pi gives winner low prob
    log_ref_w=-2.0,
    log_pi_l=-1.0,   # pi gives loser high prob
    log_ref_l=-2.0,
    beta=1.0
)

print(f"Loss when policy prefers winner: {loss1:.4f} (should be low)")
print(f"Loss when policy prefers loser:  {loss2:.4f} (should be high)")

# Batch computation
n_pairs = 100
log_pi_w = np.random.randn(n_pairs) - 2
log_ref_w = np.random.randn(n_pairs) - 2
log_pi_l = np.random.randn(n_pairs) - 2
log_ref_l = np.random.randn(n_pairs) - 2

losses = np.array([dpo_loss(log_pi_w[i], log_ref_w[i], log_pi_l[i], log_ref_l[i])
                    for i in range(n_pairs)])
print(f"\nBatch DPO loss: mean={losses.mean():.4f}, std={losses.std():.4f}")

## Exercise 8: GAE Computation

Compute Generalised Advantage Estimation for a given trajectory.

$$\hat{A}_t^{GAE} = \sum_{l=0}^{T-t-1} (\gamma\lambda)^l \delta_{t+l}$$

where $\delta_t = r_{t+1} + \gamma V(s_{t+1}) - V(s_t)$

In [ ]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """
    Compute GAE advantages.
    Args:
        rewards: list of rewards [r_1, r_2, ..., r_T]
        values: list of value estimates [V(s_0), V(s_1), ..., V(s_T)]
                (T+1 values, including terminal)
    Returns:
        advantages: array of GAE advantages
    """
    # YOUR CODE HERE
    pass

# Test
rewards = [0.0, 0.0, 0.0, 1.0, 0.0]
values = [0.5, 0.4, 0.3, 0.8, 0.1, 0.0]  # includes V(s_terminal) = 0

### Solution 8

In [ ]:
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    T = len(rewards)
    advantages = np.zeros(T)
    last_adv = 0

    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t+1] - values[t]
        advantages[t] = delta + gamma * lam * last_adv
        last_adv = advantages[t]

    return advantages

rewards = [0.0, 0.0, 0.0, 1.0, 0.0]
values = [0.5, 0.4, 0.3, 0.8, 0.1, 0.0]

for lam in [0.0, 0.5, 0.95, 1.0]:
    adv = compute_gae(rewards, values, gamma=0.99, lam=lam)
    print(f"lambda={lam:.2f}: advantages = {adv.round(4)}")

print("\nlambda=0: TD error only (one-step, low variance, high bias)")
print("lambda=1: MC-like (full return minus baseline, no bias, high variance)")

## Exercise 9: Experience Replay Buffer

Implement a replay buffer with both uniform and prioritised sampling.

**Tasks:**
1. Implement circular buffer with push/sample
2. Implement priority-based sampling
3. Compute importance sampling weights

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        # YOUR CODE HERE
    def push(self, state, action, reward, next_state, done):
        pass
    def sample(self, batch_size):
        pass
    def __len__(self):
        pass

### Solution 9

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.capacity = capacity
        self.buffer = []
        self.pos = 0

    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.pos] = (state, action, reward, next_state, done)
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        idx = np.random.choice(len(self.buffer), batch_size, replace=False)
        batch = [self.buffer[i] for i in idx]
        return (np.array([b[0] for b in batch]),
                np.array([b[1] for b in batch]),
                np.array([b[2] for b in batch]),
                np.array([b[3] for b in batch]),
                np.array([b[4] for b in batch]))

    def __len__(self):
        return len(self.buffer)

class PrioritisedReplayBuffer:
    def __init__(self, capacity, alpha=0.6, beta=0.4):
        self.capacity = capacity
        self.alpha = alpha
        self.beta = beta
        self.buffer = []
        self.priorities = np.zeros(capacity)
        self.pos = 0
        self.max_p = 1.0

    def push(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.pos] = (state, action, reward, next_state, done)
        self.priorities[self.pos] = self.max_p ** self.alpha
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size):
        n = len(self.buffer)
        probs = self.priorities[:n] / self.priorities[:n].sum()
        idx = np.random.choice(n, batch_size, p=probs, replace=False)
        weights = (n * probs[idx]) ** (-self.beta)
        weights /= weights.max()
        batch = [self.buffer[i] for i in idx]
        return (np.array([b[0] for b in batch]),
                np.array([b[1] for b in batch]),
                np.array([b[2] for b in batch]),
                np.array([b[3] for b in batch]),
                np.array([b[4] for b in batch]),
                weights, idx)

    def update_priorities(self, indices, td_errors):
        for i, td in zip(indices, td_errors):
            p = (abs(td) + 1e-6) ** self.alpha
            self.priorities[i] = p
            self.max_p = max(self.max_p, p)

    def __len__(self):
        return len(self.buffer)

# Test
buf = ReplayBuffer(1000)
for i in range(100):
    buf.push(np.random.randn(4), np.random.randint(2),
             np.random.randn(), np.random.randn(4), False)

s, a, r, ns, d = buf.sample(32)
print(f"Uniform buffer: sampled {s.shape[0]} transitions, state dim={s.shape[1]}")

pbuf = PrioritisedReplayBuffer(1000)
for i in range(100):
    pbuf.push(np.random.randn(4), np.random.randint(2),
              np.random.randn(), np.random.randn(4), False)

s, a, r, ns, d, w, idx = pbuf.sample(32)
print(f"Prioritised buffer: weights range [{w.min():.3f}, {w.max():.3f}]")

## Exercise 10: Contraction Mapping Verification

Numerically verify that the Bellman optimality operator is a gamma-contraction.

**Tasks:**
1. Apply $\mathcal{T}^*$ to two different value functions
2. Show $\|\mathcal{T}^* V_1 - \mathcal{T}^* V_2\|_\infty \le \gamma \|V_1 - V_2\|_\infty$
3. Track convergence rate

In [ ]:
# YOUR CODE HERE

### Solution 10

In [ ]:
# Simple 5-state chain MDP
n_s, n_a = 5, 2
gamma = 0.9

# Random MDP
np.random.seed(42)
P = np.random.dirichlet(np.ones(n_s), size=(n_s, n_a))  # P[s,a,s']
R = np.random.randn(n_s, n_a)

def bellman_optimality_operator(V, P, R, gamma):
    """Apply T* to V."""
    TV = np.zeros_like(V)
    for s in range(len(V)):
        q_vals = []
        for a in range(P.shape[1]):
            q = sum(P[s, a, sp] * (R[s, a] + gamma * V[sp]) for sp in range(len(V)))
            q_vals.append(q)
        TV[s] = max(q_vals)
    return TV

# Start from two random value functions
V1 = np.random.randn(n_s) * 5
V2 = np.random.randn(n_s) * 5

print(f"Initial ||V1 - V2||_inf = {np.max(np.abs(V1 - V2)):.4f}")

# Apply T* repeatedly and track contraction
for k in range(20):
    TV1 = bellman_optimality_operator(V1, P, R, gamma)
    TV2 = bellman_optimality_operator(V2, P, R, gamma)

    dist_before = np.max(np.abs(V1 - V2))
    dist_after = np.max(np.abs(TV1 - TV2))
    ratio = dist_after / (dist_before + 1e-15)

    if k < 5 or k >= 18:
        print(f"k={k:2d}: ||TV1-TV2|| = {dist_after:.6f}, "
              f"ratio = {ratio:.4f} (should be <= gamma={gamma})")

    V1, V2 = TV1, TV2

print(f"\nContraction verified: ratio <= gamma = {gamma} at every step")
print(f"Final ||V1 - V2||_inf = {np.max(np.abs(V1 - V2)):.8f} (converging to same V*)")